# CaImAn Parameter Sweep Report

Iterates over every `caiman_final*` run directory in `BASE_DIR`, loads each
`caiman_results.hdf5`, and for each run saves (into that same run directory):

- `parameter_report.png` — correlation image with accepted/rejected contours
  plus r_value / SNR / CNN score distributions
- `run_parameters.json` / `run_parameters.txt` — the CNMF parameters that
  produced that run (gSig, min_corr, min_pnr, merge_thr, evaluation
  thresholds, etc.) and the resulting accepted/rejected component counts

It also writes a cross-run summary (`parameter_sweep_summary.csv` /
`.png`) into `BASE_DIR` so the parameter sets can be compared at a glance.

In [ ]:
import json
import os

import cv2
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    cv2.setNumThreads(0)
except():
    pass

from IPython import get_ipython
try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

import caiman as cm
from caiman.source_extraction.cnmf import cnmf
from caiman.utils.visualization import get_contours

get_ipython().run_line_magic('matplotlib', 'inline')

In [ ]:
BASE_DIR = "/media/toor/Seagate3/YOUNG_MICE/Mouse946/test_cropping"

run_dirs = sorted(
    d for d in os.listdir(BASE_DIR)
    if d.startswith("caiman_final")
    and os.path.isfile(os.path.join(BASE_DIR, d, "caiman_results.hdf5"))
)
print(f"Found {len(run_dirs)} runs in {BASE_DIR}:")
for d in run_dirs:
    print(f"  - {d}")

In [ ]:
def extract_params(fit):
    """Flatten the CNMF params + acceptance counts that matter for comparing runs."""
    p = fit.params
    est = fit.estimates
    n_acc = len(est.idx_components)
    n_rej = len(est.idx_components_bad)
    return {
        "dims": tuple(int(x) for x in fit.dims),
        "fr": p.data["fr"],
        "decay_time": p.data["decay_time"],
        "dxy": tuple(p.data["dxy"]),
        "gSig": tuple(int(x) for x in p.init["gSig"]),
        "gSiz": tuple(int(x) for x in p.init["gSiz"]),
        "gSig_filt": tuple(int(x) for x in p.motion["gSig_filt"]),
        "min_corr": p.init["min_corr"],
        "min_pnr": p.init["min_pnr"],
        "nb": p.init["nb"],
        "merge_thr": p.merging["merge_thr"],
        "rf": p.patch["rf"],
        "stride": p.patch["stride"],
        "max_shifts": tuple(int(x) for x in p.motion["max_shifts"]),
        "strides": tuple(int(x) for x in p.motion["strides"]),
        "overlaps": tuple(int(x) for x in p.motion["overlaps"]),
        "pw_rigid": p.motion["pw_rigid"],
        "min_SNR": p.quality["min_SNR"],
        "SNR_lowest": p.quality["SNR_lowest"],
        "rval_thr": p.quality["rval_thr"],
        "rval_lowest": p.quality["rval_lowest"],
        "min_cnn_thr": p.quality["min_cnn_thr"],
        "cnn_lowest": p.quality["cnn_lowest"],
        "n_accepted": n_acc,
        "n_rejected": n_rej,
        "n_total": n_acc + n_rej,
    }


def plot_run_report(fit, run_label, params_summary):
    """Contour overlay (accepted=lime, rejected=red) + per-metric score histograms."""
    est = fit.estimates
    idx_acc = np.array(est.idx_components)
    idx_rej = np.array(est.idx_components_bad)
    if est.coordinates is None:
        est.coordinates = get_contours(est.A, fit.dims)

    fig = plt.figure(figsize=(20, 5))
    gs = gridspec.GridSpec(1, 4, width_ratios=[2, 1, 1, 1], wspace=0.3)

    ax0 = fig.add_subplot(gs[0])
    ax0.imshow(est.Cn, cmap="gray")
    for color, indices in [("lime", idx_acc), ("red", idx_rej)]:
        for idx in indices:
            c = est.coordinates[idx]["coordinates"]
            ax0.plot(c[:, 0], c[:, 1], color=color, linewidth=0.8, alpha=0.85)
    ax0.set_title(
        f"{run_label}\naccepted={len(idx_acc)}  rejected={len(idx_rej)}",
        fontsize=10,
    )
    ax0.axis("off")

    cnn = est.cnn_preds
    metric_data = {
        "r_value": est.r_values,
        "SNR": est.SNR_comp,
        "CNN": cnn if cnn is not None else np.full(est.A.shape[1], np.nan),
    }
    metric_axes = [fig.add_subplot(gs[i]) for i in (1, 2, 3)]
    for ax, (metric, arr) in zip(metric_axes, metric_data.items()):
        acc_vals = arr[idx_acc]
        rej_vals = arr[idx_rej]
        acc_vals = acc_vals[np.isfinite(acc_vals)]
        rej_vals = rej_vals[np.isfinite(rej_vals)]
        combined = np.concatenate([acc_vals, rej_vals]) if len(acc_vals) or len(rej_vals) else np.array([0.0, 1.0])
        bins = np.linspace(combined.min(), combined.max(), 25)
        ax.hist(rej_vals, bins=bins, color="red", alpha=0.5, label=f"rejected ({len(rej_vals)})")
        ax.hist(acc_vals, bins=bins, color="lime", alpha=0.6, label=f"accepted ({len(acc_vals)})")
        ax.set_title(metric, fontsize=10)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    param_str = (
        f"gSig={params_summary['gSig']}  gSig_filt={params_summary['gSig_filt']}  "
        f"min_corr={params_summary['min_corr']}  min_pnr={params_summary['min_pnr']}  "
        f"merge_thr={params_summary['merge_thr']}  nb={params_summary['nb']}\n"
        f"min_SNR={params_summary['min_SNR']}  rval_thr={params_summary['rval_thr']}  "
        f"min_cnn_thr={params_summary['min_cnn_thr']}  cnn_lowest={params_summary['cnn_lowest']}"
    )
    fig.suptitle(param_str, fontsize=9, y=1.05)
    plt.tight_layout()
    return fig

In [ ]:
summary_rows = []
fits_by_run = {}   # keep loaded CNMF fits around for cross-run comparison below

for run_name in run_dirs:
    run_dir = os.path.join(BASE_DIR, run_name)
    hdf5_path = os.path.join(run_dir, "caiman_results.hdf5")
    print(f"\n=== {run_name} ===")
    try:
        fit = cnmf.load_CNMF(hdf5_path, n_processes=1, dview=None)
    except Exception as e:
        print(f"  FAILED to load: {e}")
        continue

    params_summary = extract_params(fit)
    params_summary["run_name"] = run_name
    summary_rows.append(params_summary)
    fits_by_run[run_name] = fit

    with open(os.path.join(run_dir, "run_parameters.json"), "w") as f:
        json.dump(params_summary, f, indent=2, default=str)
    with open(os.path.join(run_dir, "run_parameters.txt"), "w") as f:
        for k, v in params_summary.items():
            f.write(f"{k}: {v}\n")

    fig = plot_run_report(fit, run_name, params_summary)
    fig.savefig(os.path.join(run_dir, "parameter_report.png"), dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(
        f"  accepted={params_summary['n_accepted']}  rejected={params_summary['n_rejected']}  "
        f"saved report + parameters -> {run_dir}"
    )

print(f"\nDone. Processed {len(summary_rows)}/{len(run_dirs)} runs.")

In [ ]:
df_summary = (
    pd.DataFrame(summary_rows)
    .set_index("run_name")
    .sort_values("n_accepted", ascending=False)
)

summary_csv = os.path.join(BASE_DIR, "parameter_sweep_summary.csv")
df_summary.to_csv(summary_csv)
print(f"Saved summary table -> {summary_csv}")
df_summary

In [ ]:
fig, ax = plt.subplots(figsize=(max(8, len(df_summary) * 0.9), 5))
x = np.arange(len(df_summary))
ax.bar(x, df_summary["n_accepted"], color="lime", alpha=0.8, label="accepted")
ax.bar(x, df_summary["n_rejected"], bottom=df_summary["n_accepted"], color="red", alpha=0.6, label="rejected")
ax.set_xticks(x)
ax.set_xticklabels(df_summary.index, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("# components")
ax.set_title(f"Accepted / rejected components per parameter run\n{BASE_DIR}")
ax.legend()
plt.tight_layout()

summary_png = os.path.join(BASE_DIR, "parameter_sweep_summary.png")
fig.savefig(summary_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved summary plot -> {summary_png}")

## Cross-Run Component Matching

To compare evaluation scores for the *same* underlying cell across parameter
sets, we match each run's accepted components against a reference run using
spatial footprint IoU (Hungarian algorithm) — same approach as
`compare_caiman_runs.ipynb`, generalized here to all runs in the sweep
against one auto-selected reference (the run with the most accepted
components).

In [ ]:
from scipy.optimize import linear_sum_assignment

IOU_THRESHOLD = 0.3  # min IoU to link two components as the same cell

REFERENCE_RUN_NAME = df_summary.index[0]  # run with the most accepted components
print(f"Using '{REFERENCE_RUN_NAME}' as the matching reference "
      f"({df_summary.loc[REFERENCE_RUN_NAME, 'n_accepted']} accepted components)")


def build_footprint_masks(fit, dims):
    """Returns (masks: n_acc x H x W bool, ids: 1D int array of component IDs)."""
    idx_acc = np.array(fit.estimates.idx_components)
    if len(idx_acc) == 0:
        return np.zeros((0,) + tuple(dims), dtype=bool), idx_acc
    A = fit.estimates.A
    masks = np.stack(
        [np.asarray(A[:, i].todense()).reshape(dims, order="F") > 0 for i in idx_acc],
        axis=0,
    )
    return masks, idx_acc


def iou_matrix(masks_a, masks_b):
    """(n_a x n_b) IoU matrix between two sets of H x W boolean footprint masks."""
    if len(masks_a) == 0 or len(masks_b) == 0:
        return np.zeros((len(masks_a), len(masks_b)))
    a = masks_a.reshape(len(masks_a), -1).astype(np.float32)
    b = masks_b.reshape(len(masks_b), -1).astype(np.float32)
    inter = a @ b.T
    union = a.sum(1, keepdims=True) + b.sum(1, keepdims=True).T - inter
    return np.where(union > 0, inter / union, 0.0)


def hungarian_match(iou_mat, threshold):
    """Optimal 1-to-1 matching; returns (row_idx, col_idx, iou_vals) for pairs >= threshold."""
    if iou_mat.size == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.array([])
    row_ind, col_ind = linear_sum_assignment(-iou_mat)
    vals = iou_mat[row_ind, col_ind]
    keep = vals >= threshold
    return row_ind[keep], col_ind[keep], vals[keep]


ref_fit = fits_by_run[REFERENCE_RUN_NAME]
ref_masks, ref_ids = build_footprint_masks(ref_fit, ref_fit.dims)

### Per-Component Scores vs. Reference (colored by match confidence)

For each non-reference run, matched cells are plotted as
reference-value vs. this-run-value for r_value / SNR / CNN, colored by the
IoU match confidence. Points near the diagonal are cells whose evaluation
score is stable across the parameter change; color shows how confident the
spatial match is.

In [ ]:
match_metric_labels = [("r_value", "r_values"), ("SNR", "SNR_comp"), ("CNN", "cnn_preds")]
match_stats = {}

for run_name in run_dirs:
    if run_name == REFERENCE_RUN_NAME or run_name not in fits_by_run:
        continue
    run_fit = fits_by_run[run_name]
    masks, ids = build_footprint_masks(run_fit, run_fit.dims)
    iou = iou_matrix(ref_masks, masks)
    r_idx, c_idx, conf = hungarian_match(iou, IOU_THRESHOLD)

    if len(conf) == 0:
        print(f"{run_name}: no matches >= IoU {IOU_THRESHOLD}, skipping")
        continue

    ref_comp_ids = ref_ids[r_idx]
    run_comp_ids = ids[c_idx]
    match_stats[run_name] = {"n_matched": len(conf), "mean_confidence": float(conf.mean())}

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (label, attr) in zip(axes, match_metric_labels):
        ref_arr = getattr(ref_fit.estimates, attr)
        run_arr = getattr(run_fit.estimates, attr)
        ref_vals = ref_arr[ref_comp_ids] if ref_arr is not None else np.full(len(ref_comp_ids), np.nan)
        run_vals = run_arr[run_comp_ids] if run_arr is not None else np.full(len(run_comp_ids), np.nan)
        valid = np.isfinite(ref_vals) & np.isfinite(run_vals)

        sc = ax.scatter(ref_vals[valid], run_vals[valid], c=conf[valid],
                         cmap="plasma", s=40, alpha=0.85, vmin=IOU_THRESHOLD, vmax=1.0)
        if valid.sum():
            lo = min(ref_vals[valid].min(), run_vals[valid].min())
            hi = max(ref_vals[valid].max(), run_vals[valid].max())
            pad = (hi - lo) * 0.05 or 0.05
            ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", linewidth=0.8, alpha=0.5)
        ax.set_xlabel(f"{REFERENCE_RUN_NAME}  {label}")
        ax.set_ylabel(f"{run_name}  {label}")
        ax.set_title(label)
        ax.grid(alpha=0.3)
        plt.colorbar(sc, ax=ax, label="match confidence (IoU)")

    fig.suptitle(
        f"{run_name}  vs reference '{REFERENCE_RUN_NAME}'\n"
        f"{len(conf)} matched cells (IoU >= {IOU_THRESHOLD}), "
        f"mean confidence={conf.mean():.2f}",
        fontsize=11,
    )
    plt.tight_layout()
    fig.savefig(os.path.join(BASE_DIR, run_name, "matched_scores_vs_reference.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"{run_name}: {len(conf)} matched cells, mean confidence={conf.mean():.3f}")

### Grouped Evaluation Scores — All Runs

Accepted (green) vs. rejected (red) r_value / SNR / CNN distributions for
every run in the sweep, side by side — shows how each parameter set shifts
the acceptance boundary independent of cross-run cell matching.

In [ ]:
acc_color = "#2ca02c"
rej_color = "#d62728"
metrics = [("r_value", "r_values"), ("SNR", "SNR_comp"), ("CNN", "cnn_preds")]

fig, axes = plt.subplots(1, 3, figsize=(max(16, len(run_dirs) * 1.6), 6))

positions, xlabels, xcolors = [], [], []
for run_i, run_name in enumerate(run_dirs):
    base = run_i * 3
    positions.append((base, base + 1))
    xlabels += [f"{run_name}\nacc", f"{run_name}\nrej"]
    xcolors += [acc_color, rej_color]

for ax, (label, attr) in zip(axes, metrics):
    for run_i, run_name in enumerate(run_dirs):
        if run_name not in fits_by_run:
            continue
        fit = fits_by_run[run_name]
        arr = getattr(fit.estimates, attr)
        if arr is None:
            continue
        idx_acc = np.array(fit.estimates.idx_components)
        idx_rej = np.array(fit.estimates.idx_components_bad)
        acc_data = arr[idx_acc]
        rej_data = arr[idx_rej]
        acc_data = acc_data[np.isfinite(acc_data)]
        rej_data = rej_data[np.isfinite(rej_data)]
        acc_pos, rej_pos = positions[run_i]
        for data, pos, color in [(acc_data, acc_pos, acc_color), (rej_data, rej_pos, rej_color)]:
            if len(data) < 2:
                continue
            parts = ax.violinplot([data], positions=[pos], showmedians=True, showextrema=True)
            parts["bodies"][0].set_facecolor(color)
            parts["bodies"][0].set_alpha(0.65)
            for key in ("cmedians", "cmaxes", "cmins", "cbars"):
                parts[key].set_color("black")
                parts[key].set_linewidth(1.0)
            ax.scatter(pos, np.mean(data), marker="D", color="white",
                       edgecolors="black", s=30, zorder=3, linewidths=0.7)

    for run_i in range(1, len(run_dirs)):
        ax.axvline(run_i * 3 - 0.5, color="gray", linewidth=0.8, linestyle="--", alpha=0.4)

    flat_pos = [p for pair in positions for p in pair]
    ax.set_xticks(flat_pos)
    ax.set_xticklabels(xlabels, fontsize=7, rotation=90)
    for tick, col in zip(ax.get_xticklabels(), xcolors):
        tick.set_color(col)
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.grid(axis="y", alpha=0.3)

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Patch(facecolor=acc_color, alpha=0.65, label="accepted"),
    Patch(facecolor=rej_color, alpha=0.65, label="rejected"),
    Line2D([0], [0], color="black", linewidth=1.2, label="median"),
    Line2D([0], [0], marker="D", color="w", markerfacecolor="white",
           markeredgecolor="black", markersize=7, label="mean"),
]
axes[-1].legend(handles=legend_elements, loc="upper right", fontsize=8)

fig.suptitle(f"Evaluation scores across all parameter runs\n{BASE_DIR}", fontsize=12)
plt.tight_layout()

grouped_png = os.path.join(BASE_DIR, "grouped_evaluation_scores.png")
fig.savefig(grouped_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved grouped evaluation score comparison -> {grouped_png}")

In [ ]:
# Consolidate every plot generated in this notebook into a single results/ folder,
# named after the gSig / gSig_filt parameters that produced it.
# (Several runs share the same gSig/gSig_filt, e.g. the caiman_final_58_* nb sweep,
# so the run_name is appended to keep filenames unique.)
import shutil

results_dir = os.path.join(BASE_DIR, "results")
os.makedirs(results_dir, exist_ok=True)

for row in summary_rows:
    run_name = row["run_name"]
    gx, gy = row["gSig"]
    fx, fy = row["gSig_filt"]
    tag = f"gSig_{gx}x{gy}__gSigFilt_{fx}x{fy}__{run_name}"

    src = os.path.join(BASE_DIR, run_name, "parameter_report.png")
    dst = os.path.join(results_dir, f"{tag}.png")
    shutil.copy2(src, dst)
    print(f"{run_name:25s} -> results/{os.path.basename(dst)}")

    match_src = os.path.join(BASE_DIR, run_name, "matched_scores_vs_reference.png")
    if os.path.isfile(match_src):
        match_dst = os.path.join(results_dir, f"{tag}__matched_scores.png")
        shutil.copy2(match_src, match_dst)
        print(f"{'':25s} -> results/{os.path.basename(match_dst)}")

for fname in ["parameter_sweep_summary.png", "grouped_evaluation_scores.png"]:
    src = os.path.join(BASE_DIR, fname)
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(results_dir, fname))
        print(f"(sweep-wide) -> results/{fname}")

print(f"\nAll notebook plots consolidated -> {results_dir}")